### Querying Data: American Community Survey, 5-Year Estimate Detailed Tables

#### What is `acspsuedo`?

The `acspsuedo` library was designed with a focus on the United States Census Bureau's American Community Survey (ACS) datasets, an annual survey effort conducted by the Bureau since 2005 to understand the social, economic, demographic, and housing information for households across the United States, the District of Columbia, and the Commonwealth of Puerto Rico.

This particular package seeks to make data queries and extraction as seemless as possible with a user-friendly interface designed to support a host of purposes, such as:
- Querying metadata on particular variable and/or table information offered by any one of the ACS datasets (cf. `acspsuedo.datasets`)

- Querying metadata on the types of geographic scopes at which ACS demographic data may be available at (such as at the state-level, the county-level, and so forth)

- Running data queries to the Bureau, with support for concurrent application for users interested in ETL/ELT processes

- Enabling users with Census Bureau API keys to run multiple data queries in a session

- Continuous monitoring of the Census Bureau API, in order to ensure the most up-to-date information on all ACS datasets

- Caching the Bureau's Topologically Integerated Geographic Encoding and Referencing (TIGER) shapefiles, which are designed to provide geographic representations to support maps and/or geographic analysis<sup>[1]</sup>

While we certainly cannot summarize every important facet of the Census Bureau (domain knowledge is its own skill!), we can try to make the data process easier.

Hence, `acspsuedo`!

<br>

*Footnotes*

1. For more information on TIGER Shapefiles, visit the Census Bureau's documentation: [https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/tiger-geo-line.html](https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/tiger-geo-line.html). TIGER shapefiles contain geograhic identifiers (`GEOID`), which can be used to relate to those in ACS data, effectively projecting queried demographic data from the ACS on maps supplied by Shapefiles.

The following notebook will traverse the core component of the library: `acspsuedo.query`.

Users will most frequent this module for the specific objects and interfaces that fulfill the tasks and purposes as specified above.

<br>

This notebook makes use of the ACS 5-Year Estimate Detailed Table dataset during the 2023 calendar year. Per the guidance of the Census Bureau,

> The American Community Survey (ACS) is an ongoing survey that provides data every year—giving communities the current information they need to make important decisions. The ACS covers a broad range of topics about social, economic, housing, and demographic characteristics of the U.S. population.
>
> The ACS 5-year estimates are available for the nation, all states, the District of Columbia, Puerto Rico, all congressional districts and metropolitan statistical areas, counties, places (i.e., towns or cities), ZIP Code Tabulation Areas, census tracts, and block groups
>
> - **Detailed Tables**: Tables contain the most detailed estimates on all topics for all geographies. The data are presented as estimates. Detailed Tables are available down to the block group level.

*Source: [https://www.census.gov/data/developers/data-sets/acs-5year.html](https://www.census.gov/data/developers/data-sets/acs-5year.html)*

#### What is the data we are querying?

In our case, we will be interested in querying data on median contract rents, which is provided by the `'B25058'` table from the ACS 5-Year Estimate Detailed Table dataset.

In [7]:
import acspsuedo.query as apq

from acspsuedo.datasets import ACS5 # <- Corresponds to the ACS 5-Year Estimate Detailed Table

DATASET = ACS5

YEAR = 2023 # <- Our selected calendar year

MEDIAN_CONTRACT_RENT_TABLE = 'B25058'

As one may expect, the entry-point for the `acspsuedo` package is the `query` module.

As we have also mentioned, this module is also handy for providing a set of utilties for other purposes, such as viewing metadata on our table of interest. This is a nice feature to have, as many users do not immediately know what variables and/or tables to look up but have a general idea of what information they might be interested in looking up.

In [8]:
# Viewing metadata on our selected table
apq.variable_cache.tbl_metadata_df(DATASET, YEAR, MEDIAN_CONTRACT_RENT_TABLE)

,DATASET,YEAR,VARIABLE,LABEL,VARIABLE_TYPE,TABLE,TOPIC
0,acs/acs5,2023,B25058_001E,Estimate!!Median contract rent,int,B25058,Median Contract Rent (Dollars)
1,acs/acs5,2023,B25058_001EA,Annotation of Estimate!!Median contract rent,string,B25058,Median Contract Rent (Dollars)
2,acs/acs5,2023,B25058_001M,Margin of Error!!Median contract rent,int,B25058,Median Contract Rent (Dollars)
3,acs/acs5,2023,B25058_001MA,Annotation of Margin of Error!!Median contract...,string,B25058,Median Contract Rent (Dollars)


In [10]:
# Viewing metadata on any topics pertaining to 'rents'

TOPICS = '|'.join(('rents',))

METADATA_DF = apq.variable_cache.var_metadata_df(DATASET, YEAR)

METADATA_DF[
    METADATA_DF['LABEL'].str.contains(TOPICS) |
    METADATA_DF['TOPIC'].str.contains(TOPICS)
]

,DATASET,YEAR,VARIABLE,LABEL,VARIABLE_TYPE,TABLE,TOPIC
1666,acs/acs5,2023,B05009_001E,Estimate!!Total:,int,B05009,Age And Nativity Of Own Children Under 18 Year...
1667,acs/acs5,2023,B05009_002E,Estimate!!Total:!!Under 6 years:,int,B05009,Age And Nativity Of Own Children Under 18 Year...
1668,acs/acs5,2023,B05009_003E,Estimate!!Total:!!Under 6 years:!!Living with ...,int,B05009,Age And Nativity Of Own Children Under 18 Year...
1669,acs/acs5,2023,B05009_004E,Estimate!!Total:!!Under 6 years:!!Living with ...,int,B05009,Age And Nativity Of Own Children Under 18 Year...
1670,acs/acs5,2023,B05009_005E,Estimate!!Total:!!Under 6 years:!!Living with ...,int,B05009,Age And Nativity Of Own Children Under 18 Year...
...,...,...,...,...,...,...,...
26209,acs/acs5,2023,B99102_003E,Estimate!!Total:!!Not allocated,int,B99102,Allocation Of Grandparents Living With Grandch...
26210,acs/acs5,2023,B99103_001E,Estimate!!Total:,int,B99103,Allocation Of Grandparents Responsible For Gra...
26211,acs/acs5,2023,B99103_002E,Estimate!!Total:!!Allocated,int,B99103,Allocation Of Grandparents Responsible For Gra...
26212,acs/acs5,2023,B99103_003E,Estimate!!Total:!!Not allocated,int,B99103,Allocation Of Grandparents Responsible For Gra...


*Note:*

Users interested in querying a particular genre of data (as in rents, in our case) may be interested in viewing what variables and/or tables are supported for a given dataset during a given year. While we have briefly explored querying for such metadata here through the `variable_cache` interface, a full explanation is easily provided in the [`Viewing_Metadata_ACS1_PUMS_2014`](https://github.com/ramindersinghdubb/acspsuedo/blob/main/notebooks/Viewing_Metadata_ACS1_PUMS_2014.ipynb) notebook.

While we are here, we may also want to check out what type of geographic scopes are supported for the 5-Year Detailed Tables.

In [11]:
import acspsuedo.query as apq

apq.view_geographic_paths(DATASET, YEAR)

[['us'],
 ['region'],
 ['division'],
 ['state'],
 ['state', 'county'],
 ['county'],
 ['state', 'county', 'county_subdivision'],
 ['state', 'county_subdivision'],
 ['state', 'county', 'county_subdivision', 'subminor_civil_division'],
 ['state', 'county', 'county_subdivision', 'place_remainder_or_part'],
 ['state', 'county', 'tract'],
 ['state', 'tract'],
 ['state', 'county', 'tract', 'block_group'],
 ['state', 'county', 'block_group'],
 ['state', 'place', 'county_or_part'],
 ['state', 'place'],
 ['place'],
 ['state', 'consolidated_city'],
 ['consolidated_city'],
 ['state', 'consolidated_city', 'place_or_part'],
 ['state', 'alaska_native_regional_corporation'],
 ['alaska_native_regional_corporation'],
 ['american_indian_area_alaska_native_area_hawaiian_home_land'],
 ['american_indian_area_alaska_native_area_hawaiian_home_land',
  'tribal_subdivision_remainder'],
 ['tribal_subdivision_remainder'],
 ['american_indian_area_alaska_native_area_reservation_or_statistical_entity_only'],
 ['amer

*Note:*

Users interested in querying data may be interested in viewing what geographic scopes/specifiers to supply for a desired set of data are available for a given dataset during a given year, as well as if a certain scope is supported. While we have briefly explored querying for such metadata here, a full explanation is easily provided in the [`Viewing_Geo_Metadata_ACS1_2021`](https://github.com/ramindersinghdubb/acspsuedo/blob/main/notebooks/Viewing_Geo_Metadata_ACS1_2021.ipynb) notebook.

For our purposes, we will query our table by taking the `state` and `tract` geographic specifiers.

In other words, we are projecting our rental data at the (census) tract scope restricted "above" by the state of our choosing. In our case, the state will be the state of California so that our rental data will be made available at the tract scope restricted to the state of California.

Usually, for geographic specifiers, users will need to use Federal Information Processing Series (FIPS) codes which help ensure the identification of specific geographic entities.<sup>[1]</sup>

We have went ahead and made FIPS codes available at the state, county, and place-level. These FIPS codes are easily accessible under the `acspsuedo.fips` module.<sup>[2]</sup>

Lastly, if users are specifying all possible geograhies at a certain level, such as in our case with making rental data available for *all* tract-level geographies within the state of California, users should specify the wildcard operator, `'*'`. This indicates that users are not looking for a particular geographic entity, but all geographic entities within a level.

<br>

*Footnotes*

1. See [https://www.census.gov/library/reference/code-lists/ansi.html](https://www.census.gov/library/reference/code-lists/ansi.html).

2. At the version of writing (0.2.1), incorporation of FIPS codes at other geographic scopes (region, combined statistical area, and several others) will be included in subsequent version releases.

#### Running our queries

`acspsuedo` makes use of two models to run queries to the Census Bureau: a standard/synchronous model and an asynchronous concurrent model (built around the built-in `async` library with the `aiohttp.ClientSession` context manager).

URL queries are naturally I/O bound processes, which is why this particular choice of a coroutine execution unit is very much suitable.

A standard query might look something like this.

In [12]:
import acspsuedo.query as apq
from acspsuedo.fips.states import CA # <- Abbreviation for the state of California

df = apq.download(
    dataset = DATASET,
    year = YEAR,
    tables = MEDIAN_CONTRACT_RENT_TABLE,
    # Geographic specifiers, indicating (in our case) data for all
    # tract-level geographies within the state of California
    state = CA,
    tract = '*',
)

df

,STATE,COUNTY,TRACT,YEAR,B25058_001E
0,06,001,400100,2023,3501.0
1,06,001,400200,2023,2795.0
2,06,001,400300,2023,2051.0
3,06,001,400400,2023,2431.0
4,06,001,400500,2023,2042.0
...,...,...,...,...,...
9124,06,115,040902,2023,2410.0
9125,06,115,041001,2023,905.0
9126,06,115,041002,2023,NaN
9127,06,115,041101,2023,638.0


Note that the download functions also support data queries for specific variables, in addition to the tables (as shown above). Let's extend it with some variables from the 'B25070' table, which is organized under the topic 'Gross Rent As A Percentage Of Household Income'.

In [13]:
METADATA_DF[
    METADATA_DF['VARIABLE'].str.contains('B25070')
]

,DATASET,YEAR,VARIABLE,LABEL,VARIABLE_TYPE,TABLE,TOPIC
21861,acs/acs5,2023,B25070_001E,Estimate!!Total:,int,B25070,Gross Rent As A Percentage Of Household Income...
21862,acs/acs5,2023,B25070_002E,Estimate!!Total:!!Less than 10.0 percent,int,B25070,Gross Rent As A Percentage Of Household Income...
21863,acs/acs5,2023,B25070_003E,Estimate!!Total:!!10.0 to 14.9 percent,int,B25070,Gross Rent As A Percentage Of Household Income...
21864,acs/acs5,2023,B25070_004E,Estimate!!Total:!!15.0 to 19.9 percent,int,B25070,Gross Rent As A Percentage Of Household Income...
21865,acs/acs5,2023,B25070_005E,Estimate!!Total:!!20.0 to 24.9 percent,int,B25070,Gross Rent As A Percentage Of Household Income...
21866,acs/acs5,2023,B25070_006E,Estimate!!Total:!!25.0 to 29.9 percent,int,B25070,Gross Rent As A Percentage Of Household Income...
21867,acs/acs5,2023,B25070_007E,Estimate!!Total:!!30.0 to 34.9 percent,int,B25070,Gross Rent As A Percentage Of Household Income...
21868,acs/acs5,2023,B25070_008E,Estimate!!Total:!!35.0 to 39.9 percent,int,B25070,Gross Rent As A Percentage Of Household Income...
21869,acs/acs5,2023,B25070_009E,Estimate!!Total:!!40.0 to 49.9 percent,int,B25070,Gross Rent As A Percentage Of Household Income...
21870,acs/acs5,2023,B25070_010E,Estimate!!Total:!!50.0 percent or more,int,B25070,Gross Rent As A Percentage Of Household Income...


In [16]:
RENT_BURDEN_VARIABLES = [
    'B25070_001E', # <- Estimate for the total number of renters in the geography of interest
    
    'B25070_007E', # <- Estimate for the total number of renters paying 30.0 to 34.9% of their income to gross rent
    'B25070_008E', # <- ... 35.0 to 39.9% of their income to gross rent
    'B25070_009E', # <- ... 40.0 to 49.9% of their income to gross rent
    'B25070_010E', # <- ... more than 50.0% of their income to gross rent
]

apq.download(
    dataset = DATASET,
    year = YEAR,
    variables = RENT_BURDEN_VARIABLES,
    tables = MEDIAN_CONTRACT_RENT_TABLE,
    state = CA,
    tract = '*',
)

,STATE,COUNTY,TRACT,YEAR,B25058_001E,B25070_001E,B25070_007E,B25070_008E,B25070_009E,B25070_010E
0,06,001,400100,2023,3501.0,109,0,0,8,0
1,06,001,400200,2023,2795.0,355,9,0,4,29
2,06,001,400300,2023,2051.0,1632,162,172,137,258
3,06,001,400400,2023,2431.0,827,10,9,105,119
4,06,001,400500,2023,2042.0,871,116,15,78,186
...,...,...,...,...,...,...,...,...,...,...
9124,06,115,040902,2023,2410.0,568,42,52,23,152
9125,06,115,041001,2023,905.0,302,8,21,0,100
9126,06,115,041002,2023,NaN,225,0,0,0,0
9127,06,115,041101,2023,638.0,276,47,0,11,54


With the asynchronous approach:

In [17]:
df = await apq.async_download(
    dataset = DATASET,
    year = YEAR,
    variables = RENT_BURDEN_VARIABLES,
    tables = MEDIAN_CONTRACT_RENT_TABLE,
    state = CA,
    tract = '*'
)

df

,STATE,COUNTY,TRACT,YEAR,B25058_001E,B25070_001E,B25070_007E,B25070_008E,B25070_009E,B25070_010E
0,06,001,400100,2023,3501.0,109,0,0,8,0
1,06,001,400200,2023,2795.0,355,9,0,4,29
2,06,001,400300,2023,2051.0,1632,162,172,137,258
3,06,001,400400,2023,2431.0,827,10,9,105,119
4,06,001,400500,2023,2042.0,871,116,15,78,186
...,...,...,...,...,...,...,...,...,...,...
9124,06,115,040902,2023,2410.0,568,42,52,23,152
9125,06,115,041001,2023,905.0,302,8,21,0,100
9126,06,115,041002,2023,NaN,225,0,0,0,0
9127,06,115,041101,2023,638.0,276,47,0,11,54


The asynchronous approach may be favorable under the circumstances of:

1. **Querying large amounts of data**, such as an ETL/ELT process or data pipeline


2. **A user-defined API key**, since API key allows for 500+ queries in a single session
    - To learn more about setting up your API key, check out the `API_Key.ipynb` notebook

### Customizing Query Returns

`acspsuedo.query` supports some customizations for queries, with a default settings for each.

`drop_annotation_variables` (boolean; default True)

The Census Bureau often contains supplementary attribute and margin-of-error
information for data queries. This information may be useful for users interested
in statistical testing and/or data enrichment. By indicating 'False', users can
specify that this information be available in their returned queries.

Cf. [https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html](https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html)

In [18]:
df = apq.download(
    dataset = DATASET,
    year = YEAR,
    variables = RENT_BURDEN_VARIABLES,
    tables = MEDIAN_CONTRACT_RENT_TABLE,
    drop_annotation_variables = False,
    state = CA,
    tract = '*',
)

df

,STATE,COUNTY,TRACT,YEAR,B25058_001E,B25058_001EA,B25058_001M,B25058_001MA,B25070_001E,B25070_007E,B25070_008E,B25070_009E,B25070_010E
0,06,001,400100,2023,3501.0,"3,500+",NaN,***,109,0,0,8,0
1,06,001,400200,2023,2795.0,NaN,640.0,NaN,355,9,0,4,29
2,06,001,400300,2023,2051.0,NaN,370.0,NaN,1632,162,172,137,258
3,06,001,400400,2023,2431.0,NaN,245.0,NaN,827,10,9,105,119
4,06,001,400500,2023,2042.0,NaN,222.0,NaN,871,116,15,78,186
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9124,06,115,040902,2023,2410.0,NaN,134.0,NaN,568,42,52,23,152
9125,06,115,041001,2023,905.0,NaN,413.0,NaN,302,8,21,0,100
9126,06,115,041002,2023,NaN,-,NaN,**,225,0,0,0,0
9127,06,115,041101,2023,638.0,NaN,70.0,NaN,276,47,0,11,54


*Note:*

Tables usually come with annotation variables (cf. our metadata query above), which is why the median contract table was accompanied by attribute and margin-of-error variables. However, since our list of rent burden variables didn't specify such annotation variables, they were not taken.

Nonetheless, if you wish for these annotation variables, be sure to specify them (e.g. `B25070_001EA`, `B25070_001M`, `B25070_001MA`, `B25070_002EA`, so forth) **AND** set `drop_annotation_variables` to False. Otherwise, the specified annotation variables will be automatically dropped.

`convert_to_na` (boolean; default True)

The Census Bureau may indicate particular records with unique integer and/or non-integer values
to designate some particular characteristics of queried tables, such as insufficient sample sizes
or estimates falling outside the range of an open-ended distribution. By indicating 'False',
users are indicating that none of these special values be replaced with `numpy.nan` values.

Cf. [https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html](https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html)

In [19]:
df = apq.download(
    dataset = DATASET,
    year = YEAR,
    variables = RENT_BURDEN_VARIABLES,
    tables = MEDIAN_CONTRACT_RENT_TABLE,
    convert_to_na = False,
    state = CA,
    tract = '*',
)

df

,STATE,COUNTY,TRACT,YEAR,B25058_001E,B25070_001E,B25070_007E,B25070_008E,B25070_009E,B25070_010E
0,06,001,400100,2023,3501,109,0,0,8,0
1,06,001,400200,2023,2795,355,9,0,4,29
2,06,001,400300,2023,2051,1632,162,172,137,258
3,06,001,400400,2023,2431,827,10,9,105,119
4,06,001,400500,2023,2042,871,116,15,78,186
...,...,...,...,...,...,...,...,...,...,...
9124,06,115,040902,2023,2410,568,42,52,23,152
9125,06,115,041001,2023,905,302,8,21,0,100
9126,06,115,041002,2023,-666666666,225,0,0,0,0
9127,06,115,041101,2023,638,276,47,0,11,54


*For `async_download` only*

<br>

`retry_rate` (integer; default 30)

For the case of larger queries, server-blocking may hamper how many request attempts can be successfully made with the desired return output. Thus, users may be interested in specifying a custom amount of request attempts to be made before failing by setting the `retry_rate` parameter.

<br>

`timeout_rate` (float or integer; default 0.1)

Likewise, users may be interested in pacing request attempts in the occasion that there is server-based blocking for large queries. By setting the `timeout_rate`, users are effectively specifying how many seconds should be waited in between request attempts (both successful and failed). The default is 0.1 seconds.


In [20]:
df = await apq.async_download(
    dataset = DATASET,
    year = YEAR,
    variables = RENT_BURDEN_VARIABLES,
    tables = MEDIAN_CONTRACT_RENT_TABLE,
    retry_rate = 10,
    timeout_rate = 1,
    state = CA,
    tract = '*'
)

df

,STATE,COUNTY,TRACT,YEAR,B25058_001E,B25070_001E,B25070_007E,B25070_008E,B25070_009E,B25070_010E
0,06,001,400100,2023,3501.0,109,0,0,8,0
1,06,001,400200,2023,2795.0,355,9,0,4,29
2,06,001,400300,2023,2051.0,1632,162,172,137,258
3,06,001,400400,2023,2431.0,827,10,9,105,119
4,06,001,400500,2023,2042.0,871,116,15,78,186
...,...,...,...,...,...,...,...,...,...,...
9124,06,115,040902,2023,2410.0,568,42,52,23,152
9125,06,115,041001,2023,905.0,302,8,21,0,100
9126,06,115,041002,2023,NaN,225,0,0,0,0
9127,06,115,041101,2023,638.0,276,47,0,11,54
